#### By: Peyman Shahidi
#### Created: Apr 26, 2026
#### Last Edit: Apr 26, 2026

<br>

Score **directional handoff costs** between tasks within each O\*NET occupation by querying a language model (via EDSL). For every ordered task pair $(A, B)$ with $A \ne B$, the model returns a JSON object with three keys:

- `"reasoning"` — a few sentences explaining the assessment (forced into the JSON so it can't be skipped).
- `"handoff_score"` — integer 0–10 (0 = seamless, 10 = highly disruptive).
- `"handoff_pct_of_task_time"` — non-negative number expressing handoff time as a percentage of Task A's duration.

**Output layout** under `data/computed_objects/taskHandoffCost_LLM{_sample}/`:

```
raw_outputs/
  {Occupation_Title}/
    {TaskID_A}_to_{TaskID_B}.json   ← full per-pair record (metadata + reasoning + scores)
parsed_outputs/
  {Occupation_Title}.csv             ← concatenated per-occupation CSV
```

**Why per-pair files?** They serve as a built-in completion tracker: a pair is "done" iff its `raw_outputs/{Occupation_Title}/{pair}.json` file exists. Long runs over large occupations can be interrupted and resumed safely at the pair level.

The pipeline runs in two stages:

- **Stage 1 — LLM call.** For each occupation, run the survey only on pending pairs. Save one JSON per pair containing pair metadata, reasoning, and scores.
- **Stage 2 — Post-processing.** For each occupation directory under `raw_outputs/`, concatenate all per-pair JSON files into a single CSV under `parsed_outputs/` with columns for both handoff variables.

In [19]:
#Python
import getpass
import os
import json
import numpy as np
import pandas as pd
from collections import defaultdict
import itertools
import random

## formatting number to appear comma separated and with two digits after decimal: e.g, 1000 shown as 1,000.00
pd.set_option('float_format', "{:,.2f}".format)

import matplotlib.pyplot as plt
#%matplotlib inline
#from matplotlib.legend import Legend

import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_rows', 200)

In [20]:
import subprocess

# Install caffeinate package
%pip install caffeinate

# Use macOS built-in caffeinate command for reliability
# This prevents the system from sleeping while the process is running
try:
    # Start caffeinate in the background
    caff_process = subprocess.Popen(['caffeinate', '-d'],
                                   stdout=subprocess.DEVNULL,
                                   stderr=subprocess.DEVNULL)
    print(f"Caffeinate mode ON ☕ – Device will stay awake (PID: {caff_process.pid})")
    print("System sleep is disabled while this process runs.")

    # Store the process ID for later cleanup
    caff_pid = caff_process.pid

except Exception as e:
    print(f"⚠️ Could not start caffeinate: {e}")
    print("Continuing without caffeinate - system may sleep during long processes.")
    caff_process = None
    caff_pid = None

Note: you may need to restart the kernel to use updated packages.
Caffeinate mode ON ☕ – Device will stay awake (PID: 12864)
System sleep is disabled while this process runs.


In [ ]:
main_folder_path = ".."
input_data_path = f"{main_folder_path}/data"

# Sample mode: set to a positive int to run on only that many occupations
# (writes to a separate `_sample` folder so the production output stays clean).
# In sample mode, pick the `sample_size` occupations with the FEWEST tasks
# (filtered by `max_tasks_for_sample`), so smoke tests run quickly.
# Set sample_size to None to process all occupations.
sample_size = None
max_tasks_for_sample = 10  # only used when sample_size is set

output_subfolder = 'taskHandoffCost_LLM_sample' if sample_size else 'taskHandoffCost_LLM'
output_data_path = f'{input_data_path}/computed_objects/{output_subfolder}'

# Stage 1 outputs: one folder per occupation, one JSON file per directional pair
raw_output_path = f'{output_data_path}/raw_outputs'

# Stage 2 outputs: per-occupation concatenated CSVs
parsed_output_path = f'{output_data_path}/parsed_outputs'

output_plot_path = f"{main_folder_path}/writeup/plots/{output_subfolder}"

In [22]:
# Create directories if they don't exist
for path in [raw_output_path, parsed_output_path, output_plot_path]:
    if not os.path.exists(path):
        os.makedirs(path)

# 1) Stage 1 — EDSL Setup and LLM Call

For each occupation we:
1. Build all $n(n-1)$ directional task pairs.
2. Filter out pairs already completed (the `raw_outputs/{Occupation_Title}/{pair}.json` file exists).
3. Run the EDSL survey only on the pending pairs.
4. For each response, parse the JSON and write one JSON file per pair containing pair metadata + reasoning + scores.

The model is asked to return a single JSON with three keys (`reasoning`, `handoff_score`, `handoff_pct_of_task_time`). Putting reasoning *inside* the JSON forces the model to include it; it can't be silently dropped.

If a response fails to parse, the raw output is saved to `{pair}.failed.txt` for inspection, and the pair is **not** marked as done — it will be retried on the next run.

In [23]:
import re
import ast
from edsl import QuestionFreeText, Scenario, Model
from textwrap import dedent


def _extract_json(s):
    """
    Extract a dict from a model response. Tries, in order:
      1. json.loads on the cleaned string (strict JSON).
      2. ast.literal_eval on the cleaned string (handles Python-dict syntax: single-quoted keys/values).
      3. Last balanced {...} block in the string, parsed by either of the above.
    Returns None if nothing yields a dict with the expected keys.
    """
    if not isinstance(s, str):
        return None
    cleaned = s.replace('```json', '').replace('```', '').strip()

    # Try strict JSON first
    try:
        d = json.loads(cleaned)
        if isinstance(d, dict):
            return d
    except (json.JSONDecodeError, AttributeError, TypeError):
        pass

    # Fallback 1: Python-dict literal (handles single-quoted keys/values)
    try:
        d = ast.literal_eval(cleaned)
        if isinstance(d, dict):
            return d
    except (ValueError, SyntaxError):
        pass

    # Fallback 2: find the last {...} block and retry both parsers on it
    matches = re.findall(r'\{[^{}]*\}', cleaned, flags=re.DOTALL)
    for candidate in reversed(matches):
        try:
            d = json.loads(candidate)
            if isinstance(d, dict):
                return d
        except json.JSONDecodeError:
            pass
        try:
            d = ast.literal_eval(candidate)
            if isinstance(d, dict):
                return d
        except (ValueError, SyntaxError):
            pass
    return None


def score_occupation_handoffs(occ_code, occ_title, tasks_df, raw_output_path):
    """
    For one occupation, ask the LLM to score every ordered pair of tasks (A, B), A != B.
    Resume-safe at the pair level: a pair is skipped iff its
    raw_outputs/{occ}/{pair}.json file exists.
    Failed parses go to {pair}.failed.txt and are retried on the next run.
    """
    safe_title = occ_title.replace(" ", "_").replace("/", "_")
    occ_dir = os.path.join(raw_output_path, safe_title)
    os.makedirs(occ_dir, exist_ok=True)

    if tasks_df.empty or len(tasks_df) < 2:
        print(f"   ⚠️  Fewer than 2 tasks for '{occ_title}' - skipping")
        return

    # All directional pairs
    tasks_records = tasks_df[['Task ID', 'Task Title']].to_dict('records')
    all_pairs = [
        {'task_a_id': a['Task ID'], 'task_a_title': a['Task Title'],
         'task_b_id': b['Task ID'], 'task_b_title': b['Task Title']}
        for a in tasks_records for b in tasks_records
        if a['Task ID'] != b['Task ID']
    ]

    # Skip pairs already done
    pending = []
    for p in all_pairs:
        pair_name = f"{p['task_a_id']}_to_{p['task_b_id']}"
        out_path = os.path.join(occ_dir, f"{pair_name}.json")
        if not os.path.exists(out_path):
            pending.append((p, pair_name, out_path))

    n_done = len(all_pairs) - len(pending)
    print(f"   • {len(tasks_df)} tasks → {len(all_pairs)} total pairs (already done: {n_done}, pending: {len(pending)})")

    if not pending:
        return

    scenarios = [
        Scenario({
            'occupation': occ_title,
            'task_a_title': p['task_a_title'],
            'task_b_title': p['task_b_title'],
        })
        for (p, _, _) in pending
    ]

    q_handoff = QuestionFreeText(
        question_name="handoff_assessment",
        question_text=dedent("""\
            You are an expert on workflow design for the occupation: {{ occupation }}.

            A worker has just finished this task in their workflow:
              Task A: {{ task_a_title }}

            Their work must now be handed off to a coworker, who will take over starting with this task:
              Task B: {{ task_b_title }}

            Assess this directional handoff (A → B) on two measures.

            (1) Disruption score (integer 0–10):
                How disruptive is the handoff? Consider how much accumulated context, partially-completed work, or implicit state must be transferred from the first worker to the second so that the second can pick up where the first left off.
                  - 0  = seamless handoff: no meaningful context transfer needed
                  - 10 = highly disruptive handoff: extensive context and state transfer required

            (2) Handoff time as a percentage of Task A's duration (non-negative number):
                Estimate the *additional time* the handoff itself adds, over and above the time spent doing Task A, expressed as a percentage of the time normally spent on Task A.
                  - 0    = handoff adds no time
                  - 25   = handoff adds ~25% of Task A's duration (a brief debrief, document, or status update)
                  - 100  = handoff time equals Task A's duration (e.g., walking through the entire process step by step)
                  - 200+ = handoff dominates the work

            Note that the handoff is directional: handing off from A to B may have a different cost than handing off from B to A.

            Return your answer as a single JSON object with EXACTLY these three keys:
              - "reasoning": a few sentences (string) explaining your assessment of both measures
              - "handoff_score": integer 0-10
              - "handoff_pct_of_task_time": non-negative number (no % sign)

            STRICT JSON FORMATTING RULES — your output MUST be parseable by Python's json.loads:
              - Use STRAIGHT double quotes (") for ALL keys and string values. Do NOT use single quotes ('), curly quotes (" "), or any non-ASCII quotes.
              - Inside the "reasoning" string, do NOT use double quotes to quote phrases. If you need to quote something, use single quotes (apostrophes) instead — e.g., write the phrase 'cleaning' rather than the phrase "cleaning". Apostrophes inside words (Task A's, worker's) are fine and require no escaping.
              - Use straight ASCII apostrophes (U+0027), not curly apostrophes (' ’).
              - Do not include trailing commas, comments, or any text before or after the JSON object.

            Example of the EXACT format expected:
            {"reasoning": "These tasks share little intermediate state because the equipment ends in a stable, self-contained condition; the next worker only needs a brief status check.", "handoff_score": 4, "handoff_pct_of_task_time": 20}

            Return ONLY the JSON object — nothing before or after it.
        """),
    )

    try:
        model = Model("gpt-5-mini", service_name="openai_v2", temperature=0.0, max_tokens=2000)
        results = q_handoff.by(model).by(scenarios).run(progress_bar=True)
        results_df = results.to_pandas()
    except Exception as e:
        print(f"   ❌ Error running survey: {e}")
        return

    n_failed = 0
    n_written = 0
    for (p, pair_name, out_path), raw in zip(pending, results_df['answer.handoff_assessment'].values):
        d = _extract_json(raw)

        if d is None or 'handoff_score' not in d:
            failed_path = os.path.join(occ_dir, f"{pair_name}.failed.txt")
            with open(failed_path, 'w') as f:
                f.write(raw if isinstance(raw, str) else '(empty response)')
            n_failed += 1
            continue

        # Persist a structured per-pair record (LLM fields + pair metadata, so the
        # JSON file is self-describing without needing the source ONET dataset).
        pair_record = {
            "occupation_title": occ_title,
            "soc_code": occ_code,
            "task_a": {"id": int(p['task_a_id']), "title": p['task_a_title']},
            "task_b": {"id": int(p['task_b_id']), "title": p['task_b_title']},
            "reasoning": d.get('reasoning', '') or '',
            "handoff_score": d.get('handoff_score'),
            "handoff_pct_of_task_time": d.get('handoff_pct_of_task_time'),
        }
        with open(out_path, 'w') as f:
            json.dump(pair_record, f, indent=2)

        n_written += 1

    if n_failed:
        print(f"   ⚠️  {n_failed}/{len(pending)} responses failed to parse JSON (saved as .failed.txt; will retry next run)")
    print(f"   ✅ Wrote {n_written} pair files for '{occ_title}'")

# 2) Read O*NET Dataset and Run Sample

In [24]:
# Read O*NET data
ONET = pd.read_csv(f'{input_data_path}/computed_objects/ONET_cleaned_tasks.csv')

# Drop DWA columns and dedup so each (occupation, task) appears once
ONET = ONET.drop(columns=['DWA ID', 'DWA Title']).drop_duplicates().reset_index(drop=True)
ONET = ONET[['O*NET-SOC Code', 'Occupation Title', 'Task ID', 'Task Title']].reset_index(drop=True)

# In sample mode, pick the `sample_size` occupations with the FEWEST tasks
# (filtered to those with < `max_tasks_for_sample` tasks).
if sample_size:
    task_counts = ONET.groupby(['O*NET-SOC Code', 'Occupation Title']).size().sort_values(kind='stable')
    eligible = task_counts[task_counts < max_tasks_for_sample]
    if len(eligible) < sample_size:
        print(f"⚠️  Only {len(eligible)} occupations have <{max_tasks_for_sample} tasks; taking all of them")
        sampled = eligible
    else:
        sampled = eligible.head(sample_size)
    sampled_codes = [code for (code, _title) in sampled.index]
    ONET = ONET[ONET['O*NET-SOC Code'].isin(sampled_codes)].reset_index(drop=True)
    print(f"SAMPLE MODE: {len(sampled_codes)} occupations with <{max_tasks_for_sample} tasks selected")
    for (code, title), n in sampled.items():
        print(f"   {code}  {title}  ({n} tasks → {n*(n-1)} directional pairs)")

print(f"\nNumber of unique ONET occupations in this run: {ONET['O*NET-SOC Code'].nunique():,}")
print(f"Number of unique ONET tasks in this run: {ONET['Task ID'].nunique():,}")
ONET.head()

SAMPLE MODE: 2 occupations with <10 tasks selected
   39-5093.00  Shampooers  (4 tasks → 12 directional pairs)
   47-2072.00  Pile Driver Operators  (5 tasks → 20 directional pairs)

Number of unique ONET occupations in this run: 2
Number of unique ONET tasks in this run: 9


,O*NET-SOC Code,Occupation Title,Task ID,Task Title
0,39-5093.00,Shampooers,14818,"Massage, shampoo, and condition patron's hair ..."
1,39-5093.00,Shampooers,14819,Advise patrons with chronic or potentially con...
2,39-5093.00,Shampooers,14820,"Treat scalp conditions and hair loss, using sp..."
3,39-5093.00,Shampooers,14821,Maintain treatment records.
4,47-2072.00,Pile Driver Operators,14901,Move hand and foot levers of hoisting equipmen...


In [25]:
# Process each occupation in the (sampled) dataset
occupation_groups = list(ONET.groupby(['O*NET-SOC Code', 'Occupation Title'], sort=False))

for i, ((soc_code, occ_title), group) in enumerate(occupation_groups, 1):
    tasks_df = group[['Task ID', 'Task Title']].drop_duplicates().sort_values('Task ID').reset_index(drop=True)
    print(f"\n[{i}/{len(occupation_groups)}] {occ_title}  ({soc_code})")
    score_occupation_handoffs(soc_code, occ_title, tasks_df, raw_output_path)

print('\n' + '=' * 50)
print('STAGE 1 COMPLETE — per-pair JSON saved')
print('=' * 50)
print(f'• raw_outputs: {raw_output_path}')


[1/2] Shampooers  (39-5093.00)
   • 4 tasks → 12 total pairs (already done: 0, pending: 12)


   ✅ Wrote 12 pair files for 'Shampooers'

[2/2] Pile Driver Operators  (47-2072.00)
   • 5 tasks → 20 total pairs (already done: 0, pending: 20)


   ✅ Wrote 20 pair files for 'Pile Driver Operators'

STAGE 1 COMPLETE — per-pair JSON saved
• raw_outputs: ../data/computed_objects/taskHandoffCost_LLM_sample/raw_outputs


# 3) Stage 2 — Concatenate Per-Pair JSONs into a Per-Occupation CSV

For each occupation directory under `raw_outputs/`, read every per-pair JSON file, extract the structured fields, and concatenate into a single CSV under `parsed_outputs/{Occupation_Title}.csv`. Columns:

`O*NET-SOC Code, Occupation Title, Task ID 1, Task Title 1, Task ID 2, Task Title 2, handoff_score, handoff_pct_of_task_time`

This step makes no API calls — re-running it after schema changes is essentially free.

In [26]:
import glob

occ_dirs = sorted([d for d in glob.glob(f"{raw_output_path}/*") if os.path.isdir(d)])
n_occs = 0
n_pairs_total = 0

for occ_dir in occ_dirs:
    safe_title = os.path.basename(occ_dir)
    pair_files = sorted(glob.glob(f"{occ_dir}/*.json"))  # excludes .failed.txt

    rows = []
    for pf in pair_files:
        try:
            with open(pf) as f:
                d = json.load(f)
        except (json.JSONDecodeError, OSError):
            continue
        rows.append({
            'O*NET-SOC Code': d.get('soc_code'),
            'Occupation Title': d.get('occupation_title'),
            'Task ID 1': d.get('task_a', {}).get('id'),
            'Task Title 1': d.get('task_a', {}).get('title'),
            'Task ID 2': d.get('task_b', {}).get('id'),
            'Task Title 2': d.get('task_b', {}).get('title'),
            'handoff_score': d.get('handoff_score'),
            'handoff_pct_of_task_time': d.get('handoff_pct_of_task_time'),
        })

    if not rows:
        print(f"   ⚠️  No parseable pair files in {safe_title}; skipping")
        continue

    df = pd.DataFrame(rows).sort_values(['Task ID 1', 'Task ID 2']).reset_index(drop=True)
    out_path = f"{parsed_output_path}/{safe_title}.csv"
    df.to_csv(out_path, index=False)
    n_occs += 1
    n_pairs_total += len(df)
    print(f"   ✅ {safe_title}: {len(df)} rows → {out_path}")

print(f"\n✓ Wrote {n_occs} per-occupation CSVs to {parsed_output_path}")
print(f"✓ Total rows across all files: {n_pairs_total:,}")

   ✅ Pile_Driver_Operators: 20 rows → ../data/computed_objects/taskHandoffCost_LLM_sample/parsed_outputs/Pile_Driver_Operators.csv
   ✅ Shampooers: 12 rows → ../data/computed_objects/taskHandoffCost_LLM_sample/parsed_outputs/Shampooers.csv

✓ Wrote 2 per-occupation CSVs to ../data/computed_objects/taskHandoffCost_LLM_sample/parsed_outputs
✓ Total rows across all files: 32


In [27]:
# Clean up caffeinate process
try:
    if 'caff_process' in globals() and caff_process is not None:
        caff_process.terminate()
        caff_process.wait()  # Wait for process to terminate
        print("Caffeinate mode OFF 💡 - System sleep is now enabled.")
    else:
        print("Caffeinate was not running or already stopped.")
except Exception as e:
    print(f"Note: {e}")
    print("Caffeinate process may have already ended.")

Caffeinate mode OFF 💡 - System sleep is now enabled.
